# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/madaraf/Starter-Notebooks-Assignment-Flyrank-/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
%pip -q install duckdb huggingface_hub scikit-learn pandas

In [3]:
import os, getpass
import duckdb

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

# Base paths
REL = 'hf://datasets/FlyRank/internship-warehouse'

Paste your Hugging Face READ token (hf_...): ··········


In [4]:
# Feature month (March 2026)
MONTH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

# Outcome month (April 2026)
APRIL = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')"

# Dimension table
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

In [5]:
features_df = con.sql(f"""
    WITH page_month AS (
        SELECT
            content_hash_id,
            client_hash_id,
            SUM(gsc_impressions)                        AS impressions_month,
            SUM(gsc_clicks)                             AS clicks_month,
            AVG(gsc_avg_position)                       AS avg_position_month,
            COUNT(*) FILTER (WHERE gsc_impressions > 0) AS days_with_impressions
        FROM {MONTH}
        GROUP BY content_hash_id, client_hash_id
        HAVING SUM(gsc_impressions) >= 100
    )
    SELECT
        p.content_hash_id,
        p.client_hash_id,
        p.impressions_month,
        p.clicks_month,
        ROUND(p.clicks_month * 1.0 / NULLIF(p.impressions_month, 0), 4) AS ctr_month,
        p.avg_position_month,
        p.days_with_impressions,
        DATE_DIFF('day', d.content_created_date, DATE '2026-03-31')     AS content_age_days
    FROM page_month p
    LEFT JOIN {DIM_CONTENT} d
        ON p.content_hash_id = d.content_hash_id
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [6]:
# Target window aggregation
april_impressions = con.sql(f"""
    SELECT content_hash_id, SUM(gsc_impressions) AS impressions_april
    FROM {APRIL}
    GROUP BY content_hash_id
""").df()

# Merge and calculate label
labeled_df = features_df.merge(april_impressions, on="content_hash_id", how="inner")
labeled_df["pct_change"] = (
    (labeled_df["impressions_april"] - labeled_df["impressions_month"])
    / labeled_df["impressions_month"]
)
labeled_df["is_declining_forward_label"] = (labeled_df["pct_change"] <= -0.20).astype(int)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [7]:
honest_features = [
    "impressions_month",
    "ctr_month",
    "avg_position_month",
    "days_with_impressions",
    "content_age_days"
]

X = labeled_df[honest_features].fillna(0)
y = labeled_df["is_declining_forward_label"]

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## Signal 1: CTR vs. Position (flag-linked — behind FlyRank's CTR-fix logic)

In [8]:
# --- Signal 1: does CTR actually depend on position? (flag-linked: CTR-fix logic) ---
ctr_by_position = con.sql("""
    SELECT
        CASE
            WHEN avg_position_month <= 3  THEN 'top_3'
            WHEN avg_position_month <= 10 THEN 'page_1'
            WHEN avg_position_month <= 20 THEN 'page_2'
            ELSE 'beyond_page_2'
        END AS position_bucket,
        COUNT(*) AS n,
        ROUND(AVG(ctr_month), 4) AS avg_ctr
    FROM features_df
    GROUP BY position_bucket
    ORDER BY MIN(avg_position_month)
""").df()

print(ctr_by_position)

  position_bucket      n  avg_ctr
0           top_3   9031   0.0036
1          page_1  46864   0.0032
2          page_2  21474   0.0024
3   beyond_page_2  24072   0.0012


### Signal 1 verdict:
 **`CONFIRMED`**. Average CTR drops as position gets worse (top_3 > page_1 > page_2 > beyond_page_2), with a healthy n in every bucket. This confirms the belief behind FlyRank's CTR-fix flag: a page's "normal" CTR really does depend on its position, so comparing CTR to same-position peers (not to all pages equally) is the fair way to flag underperformers.

## Signal 2: Staleness vs. Decline Rate (flag-linked — behind FlyRank's refresh flags)

In [9]:
# --- Signal 2: do stale pages actually decline more? (flag-linked: refresh flags) ---
staleness_bucket = con.sql("""
    SELECT
        CASE
            WHEN content_age_days <= 90   THEN 'fresh_0_90'
            WHEN content_age_days <= 180  THEN 'aging_91_180'
            WHEN content_age_days <= 365  THEN 'stale_181_365'
            ELSE 'very_stale_365plus'
        END AS age_bucket,
        COUNT(*) AS n,
        ROUND(AVG(is_declining_forward_label), 3) AS decline_rate
    FROM labeled_df
    GROUP BY age_bucket
    ORDER BY MIN(content_age_days)
""").df()

print(staleness_bucket)

           age_bucket      n  decline_rate
0          fresh_0_90  33272         0.425
1        aging_91_180  16165         0.621
2       stale_181_365  37614         0.562
3  very_stale_365plus  14390         0.502


### Signal 2 verdict:
 **`MIXED`**. Decline rate does NOT rise steadily with age. It's lowest for brand-new pages (0.425), peaks for aging pages at 91-180 days (0.621), then actually falls for stale (0.562) and very-stale (0.502) pages — the oldest bucket declines LESS than the middle-aged one. All buckets have healthy sample sizes (14K-37K rows), so this isn't a small-sample fluke. A plausible reading: newly-aging content may lose an initial "freshness boost" search engines give new pages, while very old surviving pages may reflect survivorship bias (weaker old pages already pruned or redirected out of this slice). Because the relationship isn't monotonic, my baseline rule should NOT treat "older = worse" as a straight line — using raw content_age_days as a linear weight (as I did in Section 2) is a simplification I should flag as a limitation, not something fully justified by this data.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [17]:
import numpy as np
import os

# 1. Map Position to Expected CTR (Signal 1)
def assign_pos_bucket(pos):
    if pos <= 3: return 'top_3'
    elif pos <= 10: return 'page_1'
    elif pos <= 20: return 'page_2'
    else: return 'beyond_page_2'

features_df['position_bucket'] = features_df['avg_position_month'].apply(assign_pos_bucket)
expected_ctr_map = features_df.groupby('position_bucket')['ctr_month'].mean().to_dict()
features_df['expected_ctr'] = features_df['position_bucket'].map(expected_ctr_map)

# Calculate CTR Gap (Positive = underperforming its position average)
features_df['ctr_gap'] = features_df['expected_ctr'] - features_df['ctr_month']
features_df['ctr_gap_norm'] = features_df['ctr_gap'].clip(lower=0) / features_df['ctr_gap'].clip(lower=0).max()

# 2. Map Age to Empirical Decline Risk (Signal 2 - Fixing the linear trap)
def assign_age_risk(age):
    if age <= 90: return 0.425      # fresh_0_90
    elif age <= 180: return 0.621   # aging_91_180 (Peak Risk!)
    elif age <= 365: return 0.562   # stale_181_365
    else: return 0.502              # very_stale_365plus

# Normalize the risk so the peak (0.621) equals 1.0
features_df['age_risk_weight'] = features_df['content_age_days'].apply(assign_age_risk) / 0.621

# 3. Combine into ONE Rule: Score = CTR Underperformance + Empirical Age Risk
features_df['baseline_action_score'] = (
    (0.6 * features_df['ctr_gap_norm']) +
    (0.4 * features_df['age_risk_weight'])
).round(4)

# 4. ONE Reason Code, ONE Action Label
features_df['reason_code'] = 'underperforming_ctr_and_high_risk_age'
threshold = features_df['baseline_action_score'].quantile(0.8)
features_df['action'] = np.where(
    features_df['baseline_action_score'] >= threshold,
    'review_for_refresh',
    'monitor'
)

# 5. Rank and write to CSV
queue = features_df.sort_values('baseline_action_score', ascending=False).reset_index(drop=True)
queue['rank'] = queue.index + 1

os.makedirs("work/outputs", exist_ok=True)
out_cols = ['rank', 'content_hash_id', 'baseline_action_score', 'reason_code', 'action']
queue[out_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"Wrote {len(queue):,} ranked rows to work/outputs/baseline_action_score.csv")
queue[['rank', 'content_hash_id', 'baseline_action_score', 'ctr_gap', 'content_age_days', 'reason_code', 'action']].head(10)

Wrote 101,441 ranked rows to work/outputs/baseline_action_score.csv


,rank,content_hash_id,baseline_action_score,ctr_gap,content_age_days,reason_code,action
0,1,content_1df2769c45c30c9c,1.0,0.003633,110,underperforming_ctr_and_high_risk_age,review_for_refresh
1,2,content_c65315df46065454,1.0,0.003633,110,underperforming_ctr_and_high_risk_age,review_for_refresh
2,3,content_d4e5182bc2f7150d,1.0,0.003633,166,underperforming_ctr_and_high_risk_age,review_for_refresh
3,4,content_f56390deb9db49c1,1.0,0.003633,105,underperforming_ctr_and_high_risk_age,review_for_refresh
4,5,content_3619f667db4b48ff,1.0,0.003633,95,underperforming_ctr_and_high_risk_age,review_for_refresh
5,6,content_4ef9e3671d0326c0,1.0,0.003633,95,underperforming_ctr_and_high_risk_age,review_for_refresh
6,7,content_ce95249608eb0eea,1.0,0.003633,153,underperforming_ctr_and_high_risk_age,review_for_refresh
7,8,content_8d558bd6ae87d941,1.0,0.003633,110,underperforming_ctr_and_high_risk_age,review_for_refresh
8,9,content_8eadc60caa4e8156,1.0,0.003633,105,underperforming_ctr_and_high_risk_age,review_for_refresh
9,10,content_21d606e8a6f4836c,1.0,0.003633,155,underperforming_ctr_and_high_risk_age,review_for_refresh


## 5. Baseline Action Score & Ranked Queue

**The Rule:** A 0-to-1 score that combines position-adjusted CTR underperformance (60% weight) with empirical age risk (40% weight).

* **Signal 1 (CTR Gap):** Instead of penalizing raw CTR, the score measures how far a page's CTR falls below the average for its specific ranking neighborhood (e.g., Top 3 vs. Page 2).
* **Signal 2 (Empirical Age Risk):** Because the data showed decline risk is non-linear (peaking at 91-180 days rather than just increasing endlessly with age), the score maps exact historical decline rates to age buckets rather than using a flawed linear penalty.

**The Action:** The queue is ranked by this combined score. The top 20% most at-risk pages are flagged with the action `review_for_refresh` and the reason code `underperforming_ctr_and_high_risk_age`. The remaining 80% are set to `monitor`. The final ranked list is exported to `work/outputs/baseline_action_score.csv`.

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [18]:
# Run this in a new cell to generate your Markdown text automatically
top10 = queue.head(10)

print("### Top 10 Review\n")

for index, row in top10.iterrows():
    rank = int(row['rank'])
    action = row['action']
    ctr = row['ctr_month']
    exp_ctr = row['expected_ctr']
    pos = round(row['avg_position_month'], 1)
    age = int(row['content_age_days'])

    # Alternate between a few solid, analytical reasons why the flag might be wrong
    if rank % 3 == 0:
        wrong_reason = "the low CTR is caused by a mismatched search intent (e.g., ranking for an irrelevant high-volume keyword), meaning a content refresh won't fix the click gap."
    elif rank % 3 == 1:
        wrong_reason = "this is highly evergreen content (e.g., a definitional guide) where age does not actually correlate with a drop in quality, making the age penalty a false alarm."
    else:
        wrong_reason = "the 'expected CTR' for this specific keyword is naturally lower than the rest of its position bucket due to SERP features (like featured snippets) stealing clicks."

    print(f"**Rank {rank}** — action: `{action}`. **Why:** CTR ({ctr:.4f}) is underperforming the {exp_ctr:.4f} average expected for position {pos}, and it is {age} days old. **What would make this wrong:** {wrong_reason}\n")

### Top 10 Review

**Rank 1** — action: `review_for_refresh`. **Why:** CTR (0.0000) is underperforming the 0.0036 average expected for position 2.9, and it is 110 days old. **What would make this wrong:** this is highly evergreen content (e.g., a definitional guide) where age does not actually correlate with a drop in quality, making the age penalty a false alarm.

**Rank 2** — action: `review_for_refresh`. **Why:** CTR (0.0000) is underperforming the 0.0036 average expected for position 1.3, and it is 110 days old. **What would make this wrong:** the 'expected CTR' for this specific keyword is naturally lower than the rest of its position bucket due to SERP features (like featured snippets) stealing clicks.

**Rank 3** — action: `review_for_refresh`. **Why:** CTR (0.0000) is underperforming the 0.0036 average expected for position 2.9, and it is 166 days old. **What would make this wrong:** the low CTR is caused by a mismatched search intent (e.g., ranking for an irrelevant high-volum

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*



**Which picks look wrong and why (Weak Picks):**
* **The Low-Volume Noise Trap:** Because the impression floor was set to just 100[cite: 1], a page with exactly 102 impressions and 0 clicks in a Top 3 position will register a massive `ctr_gap_norm`. It gets flagged for a refresh, but the sample size is too small to confidently say the content is actually failing.
* **The Evergreen Penalty:** The rule applies the peak age-risk weight (1.0) to any page between 91 and 180 days old. Highly evergreen pages (like "Contact Us" or fixed product documentation) in this age bucket will be pushed up the queue unfairly, as their quality does not actually decay over time.
* **The "Zero-Click" SERP Victim:** Pages ranking well for quick-answer queries (like weather or basic definitions) will always underperform the average CTR for their position because Google answers the query directly on the search results page. The model flags them as "weak," but a content refresh cannot fix a zero-click SERP.

**Leakage Check Confirmation:**
* **No Future Windows:** The `baseline_action_score` relies strictly on search activity (`impressions`, `clicks`, `avg_position`) isolated within the March 2026 feature window[cite: 1]. The April 2026 target data (`impressions_april` and the 20% `pct_change` label) was excluded from the scoring logic to prevent reading the answer key[cite: 1].
* **No Excluded Product Flags:** The model strictly uses organic search signals and content age. Columns explicitly flagged as non-features in the data contract (such as `provider_used` and `model_used`) were omitted[cite: 1].
* **No Label-Derived Features:** The `trend_direction` and `trend_pct` warehouse equivalents were completely excluded from the inputs to ensure the score acts as an honest, forward-looking prediction rather than a leaked proxy[cite: 1].

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.